In [7]:
import pymysql
import pandas as pd
from datetime import datetime, timedelta
from DATA.stock_invest_function import *

class USTradeExportExtractor:
    def __init__(self, host='localhost', user='root', password='your_password',
                 database='your_database', port=3306, charset='utf8mb4'):
        """데이터베이스 연결 초기화"""
        self.connection_params = {
            'host': host,
            'user': user,
            'password': password,
            'database': database,
            'port': port,
            'charset': charset
        }
        print(f"DB 연결 설정: {user}@{host}:{port}/{database}")

    def get_connection(self):
        """데이터베이스 연결 생성"""
        return pymysql.connect(**self.connection_params)

    def extract_by_date_string(self, target_date):
        """
        특정 날짜의 데이터 추출 (YYYY-MM-DD 형식)

        Parameters:
        target_date: str, 예: '2026-01-22'
        """
        conn = self.get_connection()
        try:
            query = """
            SELECT *
            FROM us_trade_export_monthly_with_forecast
            WHERE DATE(created_at) = %s
            ORDER BY hs_code, date_month_end
            """
            df = pd.read_sql(query, conn, params=[target_date])
            return df
        finally:
            conn.close()

    def extract_by_date_range(self, start_date, end_date):
        """
        날짜 범위의 데이터 추출

        Parameters:
        start_date: str, 예: '2026-01-20'
        end_date: str, 예: '2026-01-23'
        """
        conn = self.get_connection()
        try:
            query = """
            SELECT *
            FROM us_trade_export_monthly_with_forecast
            WHERE DATE(created_at) BETWEEN %s AND %s
            ORDER BY created_at, hs_code, date_month_end
            """
            df = pd.read_sql(query, conn, params=[start_date, end_date])
            return df
        finally:
            conn.close()

    def extract_latest_data(self):
        """가장 최근 created_at 날짜의 데이터만 추출"""
        conn = self.get_connection()
        try:
            # 먼저 최신 날짜 찾기
            query_latest = """
            SELECT DATE(MAX(created_at)) as latest_date
            FROM us_trade_export_monthly_with_forecast
            """
            latest_date = pd.read_sql(query_latest, conn).iloc[0, 0]

            # 최신 날짜의 데이터 추출
            query = """
            SELECT *
            FROM us_trade_export_monthly_with_forecast
            WHERE DATE(created_at) = %s
            ORDER BY hs_code, date_month_end
            """
            df = pd.read_sql(query, conn, params=[latest_date])
            return df, latest_date
        finally:
            conn.close()

    def extract_by_datetime_range(self, start_datetime, end_datetime):
        """
        시간까지 포함한 정밀한 범위 추출

        Parameters:
        start_datetime: str, 예: '2026-01-22 17:00:00'
        end_datetime: str, 예: '2026-01-22 18:00:00'
        """
        conn = self.get_connection()
        try:
            query = """
            SELECT *
            FROM us_trade_export_monthly_with_forecast
            WHERE created_at BETWEEN %s AND %s
            ORDER BY created_at, hs_code, date_month_end
            """
            df = pd.read_sql(query, conn, params=[start_datetime, end_datetime])
            return df
        finally:
            conn.close()

    def extract_recent_n_days(self, n_days=7):
        """최근 N일간의 데이터 추출"""
        conn = self.get_connection()
        try:
            query = """
            SELECT *
            FROM us_trade_export_monthly_with_forecast
            WHERE DATE(created_at) >= DATE_SUB(CURDATE(), INTERVAL %s DAY)
            ORDER BY created_at DESC, hs_code, date_month_end
            """
            df = pd.read_sql(query, conn, params=[n_days])
            return df
        finally:
            conn.close()

    def get_distinct_created_dates(self):
        """저장된 모든 고유 날짜 목록 조회"""
        conn = self.get_connection()
        try:
            query = """
            SELECT DISTINCT DATE(created_at) as created_date,
                   COUNT(*) as record_count
            FROM us_trade_export_monthly_with_forecast
            GROUP BY DATE(created_at)
            ORDER BY created_date DESC
            """
            df = pd.read_sql(query, conn)
            return df
        finally:
            conn.close()

    def extract_by_hs_code_and_date(self, hs_code, target_date):
        """특정 HS 코드와 날짜의 데이터 추출"""
        conn = self.get_connection()
        try:
            query = """
            SELECT *
            FROM us_trade_export_monthly_with_forecast
            WHERE hs_code = %s AND DATE(created_at) = %s
            ORDER BY date_month_end
            """
            df = pd.read_sql(query, conn, params=[hs_code, target_date])
            return df
        finally:
            conn.close()

    def get_all_hs_codes(self, target_date=None):
        """특정 날짜(또는 최신)의 모든 HS 코드 목록 조회"""
        conn = self.get_connection()
        try:
            if target_date:
                query = """
                SELECT DISTINCT hs_code
                FROM us_trade_export_monthly_with_forecast
                WHERE DATE(created_at) = %s
                ORDER BY hs_code
                """
                df = pd.read_sql(query, conn, params=[target_date])
            else:
                query = """
                SELECT DISTINCT hs_code
                FROM us_trade_export_monthly_with_forecast
                WHERE DATE(created_at) = (
                    SELECT DATE(MAX(created_at))
                    FROM us_trade_export_monthly_with_forecast
                )
                ORDER BY hs_code
                """
                df = pd.read_sql(query, conn)
            return df['hs_code'].tolist()
        finally:
            conn.close()




In [8]:
# 사용 예시

# db_info = {
#     'user': 'stox7412',         # 예: 'root'
#     'password': 'Apt106503!~', # 예: '1234'
#     # 'host' : '192.168.0.230',
#     'host': get_db_host(),         # 예: 'localhost' 또는 IP
#     'port': 3307,              # 기본 포트는 보통 3306
#     'database': 'investar'        # 예: 'trade_data'
# }


extractor = USTradeExportExtractor(
    host=get_db_host(),
    user='stox7412',
    password='Apt106503!~',
    database='investar',
    port=3307
)

# 1. 특정 날짜 데이터 추출
print("=== 특정 날짜 데이터 추출 ===")
df_today = extractor.extract_by_date_string('2026-01-22')
print(f"추출된 레코드 수: {len(df_today)}")
print(df_today.head())

# # 2. 가장 최근 데이터 추출
# print("\n=== 최신 데이터 추출 ===")
# df_latest, latest_date = extractor.extract_latest_data()
# print(f"최신 날짜: {latest_date}")
# print(f"추출된 레코드 수: {len(df_latest)}")
#
# # 3. 날짜 범위로 추출
# print("\n=== 날짜 범위 데이터 추출 ===")
# df_range = extractor.extract_by_date_range('2026-01-20', '2026-01-23')
# print(f"추출된 레코드 수: {len(df_range)}")
#
# # 4. 최근 7일 데이터
# print("\n=== 최근 7일 데이터 ===")
# df_week = extractor.extract_recent_n_days(7)
# print(f"추출된 레코드 수: {len(df_week)}")
#
# # 5. 저장된 날짜 목록 확인
# print("\n=== 저장된 날짜 목록 ===")
# dates_df = extractor.get_distinct_created_dates()
# print(dates_df)
#
# # 6. 특정 HS 코드와 날짜로 추출
# print("\n=== 특정 HS 코드 데이터 ===")
# df_hs = extractor.extract_by_hs_code_and_date('100199', '2026-01-22')
# print(f"추출된 레코드 수: {len(df_hs)}")

# 7. CSV로 저장
# df_latest.to_csv('latest_export_forecast.csv', index=False, encoding='utf-8-sig')
# print("\n최신 데이터를 CSV로 저장했습니다.")

DB 연결 설정: stox7412@192.168.0.230:3307/investar
=== 특정 날짜 데이터 추출 ===
추출된 레코드 수: 78076
  hs_code date_month_end       expDlr  expDlr_forecast  is_forecast  \
0  100199     2013-01-31  738003507.0      738003507.0            0   
1  100199     2013-02-28  844245600.0      844245600.0            0   
2  100199     2013-03-31  936336637.0      936336637.0            0   
3  100199     2013-04-30  952592398.0      952592398.0            0   
4  100199     2013-05-31  804767583.0      804767583.0            0   

                                              params          created_at  
0  {"model": "SARIMA", "order": [0, 1, 2], "seaso... 2026-01-22 17:48:20  
1  {"model": "SARIMA", "order": [0, 1, 2], "seaso... 2026-01-22 17:48:20  
2  {"model": "SARIMA", "order": [0, 1, 2], "seaso... 2026-01-22 17:48:20  
3  {"model": "SARIMA", "order": [0, 1, 2], "seaso... 2026-01-22 17:48:20  
4  {"model": "SARIMA", "order": [0, 1, 2], "seaso... 2026-01-22 17:48:20  


In [10]:
len(df_today['hs_code'].unique())

478

In [13]:
df_today[df_today['hs_code'] == '854231']

,hs_code,date_month_end,expDlr,expDlr_forecast,is_forecast,params,created_at
58471,854231,2013-01-31,1.332840e+09,1.332840e+09,0,"{""model"": ""SARIMA"", ""order"": [0, 1, 2], ""seaso...",2026-01-22 17:48:20
58472,854231,2013-02-28,1.182552e+09,1.182552e+09,0,"{""model"": ""SARIMA"", ""order"": [0, 1, 2], ""seaso...",2026-01-22 17:48:20
58473,854231,2013-03-31,1.431329e+09,1.431329e+09,0,"{""model"": ""SARIMA"", ""order"": [0, 1, 2], ""seaso...",2026-01-22 17:48:20
58474,854231,2013-04-30,1.315757e+09,1.315757e+09,0,"{""model"": ""SARIMA"", ""order"": [0, 1, 2], ""seaso...",2026-01-22 17:48:20
58475,854231,2013-05-31,1.460935e+09,1.460935e+09,0,"{""model"": ""SARIMA"", ""order"": [0, 1, 2], ""seaso...",2026-01-22 17:48:20
...,...,...,...,...,...,...,...
58638,854231,2026-12-31,NaN,2.903523e+09,1,"{""model"": ""SARIMA"", ""order"": [0, 1, 2], ""seaso...",2026-01-22 17:48:20
58639,854231,2027-01-31,NaN,3.077330e+09,1,"{""model"": ""SARIMA"", ""order"": [0, 1, 2], ""seaso...",2026-01-22 17:48:20
58640,854231,2027-02-28,NaN,2.688633e+09,1,"{""model"": ""SARIMA"", ""order"": [0, 1, 2], ""seaso...",2026-01-22 17:48:20
58641,854231,2027-03-31,NaN,2.882845e+09,1,"{""model"": ""SARIMA"", ""order"": [0, 1, 2], ""seaso...",2026-01-22 17:48:20
